# 🦙 Step 1: SFT Baseline ด้วย Unsloth 🟢

**ระดับ:** เริ่มต้น | **GPU:** T4 16GB (Colab Free) | **เวลา:** ~30 นาที

## สิ่งที่จะได้เรียน
1. โหลดโมเดล Llama-3 8B แบบ 4-bit quantization ด้วย Unsloth
2. ตั้งค่า LoRA Adapter
3. โหลด Dataset และ Format ข้อมูล
4. รัน SFTTrainer
5. ทดสอบโมเดลที่เทรนแล้ว

---
> ⚡ **เปิด GPU ก่อน!** Runtime → Change runtime type → T4 GPU

## 📦 ติดตั้ง Dependencies

Unsloth ต้องติดตั้งแยกก่อนเสมอ เพราะมี CUDA kernel พิเศษที่ต้อง compile ให้ตรงกับ environment

In [ ]:
# ติดตั้ง Unsloth (สำหรับ Colab)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q
!pip install datasets -q
print('✅ ติดตั้งเสร็จสิ้น')

## 🖥️ ตรวจสอบ GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram = round(gpu.total_memory / 1024**3, 1)
    print(f'✅ GPU: {gpu.name}')
    print(f'💾 VRAM: {vram} GB')
    if vram < 14:
        print('⚠️  VRAM น้อยกว่า 14GB → ใช้ QLoRA (load_in_4bit=True) เท่านั้น')
else:
    print('❌ ไม่พบ GPU! ไปที่ Runtime → Change runtime type → T4 GPU')

## 📦 Import Libraries

In [ ]:
from unsloth import FastLanguageModel  # หัวใจหลัก: โหลดโมเดลแบบ 4-bit อย่างรวดเร็ว
from trl import SFTTrainer             # Trainer สำหรับ Supervised Fine-Tuning
from transformers import TrainingArguments
from datasets import load_dataset
import torch

## ⚙️ ตั้งค่าพารามิเตอร์หลัก

| พารามิเตอร์ | ค่า | คำอธิบาย |
|------------|-----|----------|
| `MAX_SEQ_LENGTH` | 2048 | ความยาว token สูงสุดต่อ 1 ตัวอย่าง |
| `LOAD_IN_4BIT` | True | โหลดแบบ 4-bit = QLoRA (ประหยัด VRAM 75%) |

In [ ]:
# ชื่อโมเดลบน HuggingFace Hub
# เปลี่ยนได้เป็น: "unsloth/mistral-7b-bnb-4bit", "unsloth/gemma-7b-bnb-4bit"
MODEL_NAME = "unsloth/llama-3-8b-bnb-4bit"

# max_seq_length: ความยาวสูงสุดของ token ที่โมเดลจะรับได้ต่อ 1 ตัวอย่าง
# - ยิ่งสูง = จำบริบทได้มากขึ้น แต่กินหน่วยความจำมากขึ้น
# - T4 16GB แนะนำ 2048 | ถ้า OOM ให้ลดเหลือ 1024
MAX_SEQ_LENGTH = 2048

# dtype: None = ให้ Unsloth เลือกอัตโนมัติ (แนะนำ)
DTYPE = None

# load_in_4bit=True → QLoRA: ประหยัด VRAM ~75% (8B model ใช้แค่ ~5GB แทน ~16GB)
LOAD_IN_4BIT = True

## 🚀 โหลดโมเดลและ Tokenizer

ครั้งแรกจะใช้เวลา 2-5 นาทีในการดาวน์โหลด (~5GB)

In [ ]:
print('📥 กำลังโหลดโมเดล... (อาจใช้เวลา 2-5 นาทีครั้งแรก)')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    # token="hf_...",  # ใส่ HuggingFace token ถ้าโมเดลต้องการ login
)

print(f'✅ โหลดโมเดลสำเร็จ: {MODEL_NAME}')

## 🔧 ตั้งค่า LoRA Adapter

### LoRA คืออะไร?
แทนที่จะ Fine-tune พารามิเตอร์ทั้งหมด (8B params = ~16GB VRAM)
LoRA เพิ่ม **adapter เล็กๆ** เข้าไปในโมเดล:

```
W_new = W_frozen + (alpha/r) × A × B
```

| พารามิเตอร์ | ความหมาย | ค่าแนะนำ |
|------------|----------|----------|
| `r` (rank) | ขนาด LoRA matrix — ยิ่งสูง เรียนรู้ได้มากขึ้น แต่กิน VRAM มากขึ้น | 16 |
| `lora_alpha` | Scaling factor — มักตั้งเป็น 2×r | 32 |
| `target_modules` | Layer ที่จะใส่ LoRA | `"all-linear"` |
| `use_gradient_checkpointing` | ประหยัด VRAM เพิ่ม 30% | `"unsloth"` |

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                          # rank ของ LoRA
    lora_alpha=32,                 # scaling factor (แนะนำ = 2 * r)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],,   # ใส่ LoRA ทุก linear layer
    lora_dropout=0,                # 0 = ไม่ใช้ dropout (เร็วกว่า)
    bias="none",                   # ไม่เทรน bias
    use_gradient_checkpointing="unsloth",  # ประหยัด VRAM
    random_state=42,               # seed สำหรับ reproducibility
)

# แสดงจำนวน parameters ที่จะเทรน
# ตัวอย่างผลลัพธ์: trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.52%
model.print_trainable_parameters()

## 📊 โหลดและ Format Dataset

ใช้ **Alpaca format** (instruction-following):
```json
{
  "instruction": "คำสั่งหรือคำถาม",
  "input": "ข้อมูลเพิ่มเติม (ถ้ามี)",
  "output": "คำตอบที่ต้องการ"
}
```

> ⚠️ **EOS_TOKEN สำคัญมาก!** ต้องใส่ท้ายทุก example ไม่งั้นโมเดลจะไม่รู้จักจุดสิ้นสุด

In [ ]:
# โหลด dataset จาก HuggingFace Hub
# "yahma/alpaca-cleaned" = Alpaca dataset ที่ clean แล้ว ~52K ตัวอย่าง
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

EOS_TOKEN = tokenizer.eos_token  # </s> หรือ <|end_of_text|>

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""


def format_prompts(examples):
    """แปลง dataset แต่ละแถวให้เป็น text ตาม Alpaca format"""
    texts = []
    for instruction, inp, output in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        text = ALPACA_PROMPT.format(instruction, inp, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}


# Apply formatting ให้ทั้ง dataset (batched=True = เร็วกว่า)
dataset = dataset.map(format_prompts, batched=True)

print(f'✅ Dataset โหลดสำเร็จ: {len(dataset)} ตัวอย่าง')
print('\n📝 ตัวอย่างข้อมูล:')
print(dataset[0]["text"][:500])

## 🏋️ ตั้งค่าและรัน SFTTrainer

| พารามิเตอร์ | ค่า | คำอธิบาย |
|------------|-----|----------|
| `per_device_train_batch_size` | 2 | samples ต่อ batch ต่อ GPU (ลดเหลือ 1 ถ้า OOM) |
| `gradient_accumulation_steps` | 4 | effective batch = 2×4 = 8 |
| `max_steps` | 100 | เปลี่ยนเป็น -1 เพื่อเทรนจนครบ epoch |
| `learning_rate` | 2e-4 | LoRA ใช้ lr สูงกว่า full fine-tune ได้ |
| `optim` | adamw_8bit | AdamW แบบ 8-bit ประหยัด VRAM |

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # effective batch = 2×4 = 8
        warmup_steps=5,
        max_steps=100,           # เปลี่ยนเป็น -1 เพื่อเทรนจนครบ epoch
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        logging_steps=10,
        output_dir="outputs",
        save_strategy="steps",
        save_steps=50,
        seed=42,
        report_to="none",
    ),
)

# แสดงสถิติ GPU ก่อนเทรน
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f'🖥️  GPU: {gpu_stats.name}')
print(f'💾 VRAM ที่ใช้อยู่: {start_gpu_memory} GB / {max_memory} GB')

In [ ]:
print('🏋️  เริ่มการเทรน...')
trainer_stats = trainer.train()

end_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f'\n✅ เทรนเสร็จสิ้น!')
print(f'⏱️  เวลาที่ใช้: {trainer_stats.metrics["train_runtime"]:.2f} วินาที')
print(f'💾 VRAM สูงสุดที่ใช้: {end_gpu_memory} GB')

## 💾 บันทึก LoRA Weights

บันทึกเฉพาะ LoRA adapter (~100MB) ไม่ใช่ทั้งโมเดล (~5GB)

In [ ]:
model.save_pretrained("outputs/lora_model")
tokenizer.save_pretrained("outputs/lora_model")
print('✅ บันทึกสำเร็จที่ outputs/lora_model/')

## 🧪 ทดสอบโมเดลที่เทรนแล้ว

เปลี่ยน `test_instruction` เป็นคำถามที่ต้องการทดสอบได้เลย

In [ ]:
FastLanguageModel.for_inference(model)  # เปิด inference mode (เร็วกว่า training mode)

# 🔧 เปลี่ยนคำถามได้ที่นี่
test_instruction = "อธิบายความแตกต่างระหว่าง Machine Learning และ Deep Learning"

test_prompt = ALPACA_PROMPT.format(test_instruction, "", "")
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,      # จำนวน token สูงสุดที่จะ generate
    temperature=0.7,         # ความ "สร้างสรรค์" (0=deterministic, 1=random)
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print('📤 คำตอบจากโมเดล:')
print('─' * 50)
print(response[len(test_prompt):])
print('─' * 50)
print('\n🎉 Step 1 เสร็จสมบูรณ์! ไปต่อที่ Step 2 ได้เลย')